# APD benchmark — 0. Coordinator setup (run once)

This notebook makes the **one** shared split from the lab's `video` folder, audits recording conditions, and prepares a read-only `apd_benchmark` folder for the three operators. Nothing is uploaded: Drive is mounted and files are copied inside Drive.

**Before you start**
1. In Google Drive open *Shared with me* → `video` → ⋮ → **Organize → Add shortcut** → *My Drive*. (Colab can only see My Drive; the shortcut makes the lab folder visible at `/content/drive/MyDrive/video`.)
2. Runtime → Change runtime type → **CPU** is enough.
3. Optional: add a Colab secret `GITHUB_TOKEN` (🔑 icon on the left) with read access to `whiteraay/research`. Without it, upload the repo zip once to `MyDrive/apd_benchmark/code/research` (unzipped) yourself.

In [ ]:
#@title 1. Mount Drive and load the code
# Mount Google Drive (NOT upload) and get the project code.
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, shutil, subprocess
REPO = "/content/research"
if not os.path.exists(REPO):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # optional Colab secret (key icon on the left)
    except Exception:
        pass
    if token:
        subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/whiteraay/research.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/whiteraay/research.git"])
        print("code: cloned from GitHub")
    else:
        cands = ["/content/drive/MyDrive/apd_benchmark/code/research",
                 *glob.glob("/content/drive/.shortcut-targets-by-id/*/apd_benchmark/code/research")]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            raise SystemExit("No GITHUB_TOKEN secret and no code copy at MyDrive/apd_benchmark/code/research. "
                             "Add the apd_benchmark shortcut to My Drive (operators) or add a GITHUB_TOKEN secret.")
        shutil.copytree(src, REPO)
        print("code: copied from", src)
sys.path.insert(0, REPO + "/src")
from apd_bench import colab as apd

## 2. Look before copying
Shows how many videos and participants each cohort has, and how filenames are grouped into participants. **Check the participant ids.** If they look wrong (e.g. every file is its own participant, or the same id appears in both cohorts), change `PARTICIPANT_REGEX` and run again: group 1 of the regex must capture the participant id. Files inside a sub-folder per participant are grouped by that sub-folder automatically.

In [ ]:
#@title 2. Dry run
PARTICIPANT_REGEX = r"^([^_]+)_"     # default: text before the first "_" in the file name
print("source folder:", apd.find_source_folder())
apd.coordinator_dry_run(participant_regex=PARTICIPANT_REGEX)

## 3. Make the split
Participant-grouped, stratified by cohort, 70% train / 30% test, anonymised hashed file names. Copies run inside Drive and can take a while for many GB; if Colab disconnects, run cells 1 and 3 again — finished files are skipped.

Shared output: `MyDrive/apd_benchmark/data/`. **Private** output (never share): `MyDrive/apd_private/test_labels.csv`, `split_manifest.csv`.

In [ ]:
#@title 3. Split (copies files inside Drive)
TEST_FRAC = 0.3
SEED = 0
apd.coordinator_split(test_frac=TEST_FRAC, seed=SEED, participant_regex=PARTICIPANT_REGEX)

## 4. Recording-condition audit
Label-blind: duration, resolution, lighting, face-detection rate and audio SNR per video, plus the tertiles every model must use. Resumable.

In [ ]:
#@title 4. Audit
audit = apd.coordinator_audit()
import pandas as pd
a = pd.read_csv(audit)
print(a[["duration_s", "mean_luminance", "face_detect_rate", "audio_snr_db"]].describe().round(2))
print("rows with errors:", (a["error"].fillna("") != "").sum())

## 5. Confound check (coordinator only)
Does any recording variable separate the cohorts on its own in the training set? If yes, write it down now: it is exactly what the confound floor will measure later.

In [ ]:
#@title 5. Does recording quality predict the cohort?
lab = pd.read_csv(apd.BENCH / "data/train/train_labels.csv", dtype={"session_id": str})
m = lab.merge(a, on="session_id")
from sklearn.metrics import roc_auc_score
for c in ["duration_s", "mean_luminance", "face_detect_rate", "audio_snr_db", "speech_fraction"]:
    ok = m[c].notna()
    if ok.sum() > 5:
        auc = roc_auc_score((m.cohort[ok] == "elevated").astype(int), m[c][ok])
        print(f"{c:18s} AUROC alone = {max(auc, 1 - auc):.2f}")

In [ ]:
#@title 6. Publish code + preflight
apd.coordinator_publish_code()     # copy of the repo for operators without a GitHub token
issues = apd.coordinator_preflight()

## 7. Share and lock down
1. In Drive: right-click **`apd_benchmark`** → Share → add the three operators as **Viewer** (read-only).
2. **Do not share** `apd_private`. If the operators will use *this same Google account*, download both private files now (cell below), delete `apd_private` from Drive, and empty the Drive trash.
3. Remove the `video` shortcut from My Drive (right-click → Remove). The original folder stays shared with the lab; operators' notebooks refuse to start if they can see it.
4. Send each operator their runbook (`runbook/OPERATOR_*.md`).

In [ ]:
#@title 8. (Only if operators share this account) download private files
from google.colab import files
for f in ["test_labels.csv", "split_manifest.csv"]:
    files.download(str(apd.PRIVATE / f))